# 📈 S&P 500 Pairs Trading 交易期 (Trading Period) 策略邏輯與公式詳解

## 📝 概述
在配對交易中，**交易期 (Trading Period)**（預設 $T = 126$ 天）的核心任務是**根據 Spread Z-Score 訊號執行交易，並結合多層風控管理資金**。

> [!IMPORTANT]
> 本文件以 `strategies/trading/` 下實際運行的 `.py` 原始碼為唯一依據。已封存交易模組（`drl_lstm_trading.py`、`drl_fqi_trading.py`、`kalman_trading.py`）與從未進入策略清單的孤兒模組（`pure_dtw_trading.py`、`drl_lstm_v2_trading.py`）已於 2026-07-05 移至 `archive/trading/`，完整診斷見 `archive/config_archived_strategies.py` docstring，本文件僅在相關段落摘要引用。

### 策略池（`strategies/config.py` `strategies_raw_all`，共 10 個）：

| # | 策略 | 形成期模組 | 交易期模組 | 角色 |
| :---: | :--- | :--- | :--- | :--- |
| 1 | SSD Rolling | `ssd_rolling.py` | `zscore_trading.py` | SSD 家族基準 |
| 2 | DTW Paper Fixed (DTW) | 借用配對 | `zscore_trading.py` | 誠實 DTW 基準 |
| 3 | SSD-DTW-PCA Paper Fixed | 借用配對 | `zscore_trading.py` | 全組最佳誠實基準 |
| 4 | HDBSCAN Cluster SSD-DTW-PCA | `HDBSCAN_Cluster_SSD_DTW.py` | `zscore_trading.py` | 分組消融（vs #3） |
| 5 | SSD Rolling DRL THR | 借用 #1 配對 | `drl_threshold_trading.py` | DRL 疊加對照組 |
| 6 | HDBSCAN Cluster SSD-DTW-PCA DRL THR | 借用 #4 配對 | `drl_threshold_trading.py` | DRL 疊加實驗組 |
| 7 | HDBSCAN Cluster SSD-DTW-PCA PCA5 | `HDBSCAN_Cluster_SSD_DTW.py` | `zscore_trading.py` | 維度詛咒修復版 |
| 8 | HDBSCAN Cluster SSD-DTW-PCA PCA5 DRL THR | 借用 #7 配對 | `drl_threshold_trading.py` | DRL 疊加次高 Sharpe |
| 9 | Agglomerative Fundamentals | `agglomerative_fundamentals.py` | `zscore_trading.py` | 最佳「ML 配對」Z-Score 基準 |
| 10 | Agglomerative Fundamentals DRL THR | 借用 #9 配對 | `drl_threshold_trading.py` | 全組最佳年化報酬 |

`strategies_raw = strategies_raw_all[:]` 決定實際執行範圍；免改檔可用環境變數 `STRATEGIES_SLICE`（如 `STRATEGIES_SLICE="5:7"` 只跑 #5–#6）。

### 📂 交易期模組架構：

```
zscore_trading.py            ← #1–#4、#7、#9 使用（Z-Score 狀態機，基礎類）
drl_threshold_trading.py     ← #5、#6、#8、#10 使用（DRL 門檻選擇式 v4，繼承其風控外層）

（非現役模組已移至 archive/trading/：
  已封存 — drl_lstm_trading.py v1、drl_fqi_trading.py v3 FQI、kalman_trading.py
  孤兒（無回測數據）— pure_dtw_trading.py、drl_lstm_v2_trading.py）
```

### 回測參數（`strategies/config.py` `base_params`）：

| 參數 | 值 |
| :--- | :--- |
| `entry_z` | 2.0（進場閾值） |
| `exit_z` | 0.0（出場閾值，回歸至均值） |
| `max_holding_days` | 30 天（超時強制平倉） |
| `fee_rate` + `slippage_rate` | 各 0.001（單程各 0.1%，來回總摩擦 0.4%） |
| `zscore_window` | 0（靜態形成期參數，不滾動更新） |
| `stop_loss_pct` | 0.0（網格搜尋 `[0.0, 0.05, 0.15]`） |
| `portfolio_stop_loss_pct` | 0.0（預設不啟用全局停損） |
| `top_n_list` | `[1, 3, 5, 10, 20]`（網格搜尋） |
| `use_vol_adjust` | False（預設不啟用波動率自適應調節） |
| `INITIAL_CAPITAL` | 10,000（每配對初始資金） |

## 🧠 一、 Z-Score 狀態機核心邏輯 (`zscore_trading.py`)

所有 10 個現役策略的 Spread 重建與 Z-Score 計算均通過此模組完成（DRL-THR 模組內部的 `z_of()` 亦採用完全相同的公式，只是換了決策層）。

### 1.1 Spread 重建：現役策略統一走「路徑 B」（Z-Score 標準化對數空間）

`run_trading.py` 依 `Formation_Params` 中是否有 `OLS_Alpha` 及 `ignore_ols_alpha` 設定決定路徑。**所有現役策略的 `params` 均不設定 `OLS_Alpha` 生效或顯式設 `ignore_ols_alpha=True`**（見下方策略對照表），因此統一使用路徑 B：

$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_{\ln P_i}^{form}}{\sigma_{\ln P_i}^{form}}, \qquad \text{Spread}_t = P'_{A,t} - \beta \cdot P'_{B,t}$$

其中形成期均值/標準差（`Log_Mean_A/B`、`Log_Std_A/B`）在整個交易期保持不變（靜態模式 `zscore_window=0`）。

| 策略 | `Formation_Params` 是否含 `OLS_Alpha` | `ignore_ols_alpha` | 實際路徑 |
| :--- | :---: | :---: | :---: |
| SSD Rolling (#1) | 否 | — | B（原生不輸出） |
| DTW Paper Fixed (#2/#3) | 是（借用原版配對） | `True` | B（強制忽略） |
| HDBSCAN Cluster SSD-DTW-PCA (#4/#7) | 是 | `True` | B（強制忽略） |
| Agglomerative Fundamentals (#9) | 否 | — | B（原生不輸出，複用 SSD Rolling 排序） |

> **路徑 A**（原始 log-price OLS 殘差空間，$\text{Spread}_t = \ln P_{A,t} - \alpha - \beta \ln P_{B,t}$）目前**無現役策略使用**——它是已封存 HDBSCAN 舊特徵系（`HDBSCAN_UMAP.py`/`HDBSCAN_MultiScale.py`）與 DTW Paper 原版（座標系 artifact 版）的路徑，程式碼保留於 `zscore_trading.py._compute_spread()` 供向下相容與封存策略復活之用。
> **路徑 B1**（累積回報比值空間，`First_Price_A/B`）目前也無現役策略使用——它是已封存 `ssd_basic.py` 的路徑。

### 1.2 Z-Score 計算（靜態模式 `zscore_window=0`）

$$Z_t = \text{clip}\left(\frac{\text{Spread}_t - \mu_\epsilon}{\max(\sigma_\epsilon,\, \sigma_\text{min})},\; -10,\; 10\right)$$

若啟用波動率自適應（`use_vol_adjust=True`，目前所有現役策略預設關閉）：

$$\sigma_\text{adj} = \max\!\left(\sigma_\epsilon \cdot \max\!\left(1,\, \frac{\sigma_{20}}{\sigma_\epsilon}\right),\; \sigma_\text{min}\right)$$

### 1.3 進出場訊號（Z-Score 狀態機）

| 條件 | 動作 |
| :--- | :--- |
| $Z_t > \text{entry\_z}$（= 2.0） | **空頭建倉**：空 Ticker_A，多 Ticker_B |
| $Z_t < -\text{entry\_z}$ | **多頭建倉**：多 Ticker_A，空 Ticker_B |
| $Z_t \le \text{exit\_z}$（= 0.0）且原為空頭 | **平倉** |
| $Z_t \ge -\text{exit\_z}$ 且原為多頭 | **平倉** |
| 持倉超過 `max_holding_days`（= 30）天 | **超時強制平倉** |

### 1.4 資金部位配置（風險中性加權）與 PnL

$$W_{total} = 1.0 + |\beta|, \quad v_A = C_{pair} \cdot \frac{1.0}{W_{total}}, \quad v_B = C_{pair} \cdot \frac{|\beta|}{W_{total}}$$

$$\text{Trade PnL} = \underbrace{n_A (P_{A,t}-P_{A,\text{entry}}) + n_B (P_{B,t}-P_{B,\text{entry}})}_{\text{Raw Unrealized}} - \text{Entry Fee} - \text{Exit Fee}, \quad f_\text{total} = \text{fee\_rate} + \text{slippage\_rate}$$

## 🛡️ 二、 共享部位管理與六大風控機制 (`zscore_trading.py`)

`zscore_trading.Trading` 是所有交易模組的基礎類；`drl_threshold_trading.Trading` 是獨立實作但共用完全相同的部位配置與 PnL 會計公式（見下節），兩者共享以下風控設計原則：

| # | 機制名稱 | 觸發條件 | 行為 |
| :---: | :--- | :--- | :--- |
| 1 | **個股停損 SL** | 個配對未實現虧損 / $C_{pair}$ ≥ `stop_loss_pct`（網格 `[0, 0.05, 0.15]`） | 立即強平該配對，進入冷卻 |
| 2 | **動態 Z 發散停損 DSZ** | $\|Z_t\| >$ `dynamic_stop_z` | 判定結構性破裂，強平並凍結 |
| 3 | **全域組合停損 PSL** | 總未實現虧損 / 初始資金 ≥ `portfolio_stop_loss_pct` | 斬倉**所有**持倉配對，重置資金計數 |
| 4 | **產業分散上限 MSR** | 單一產業配對數超過 $\max(1, \lfloor N \times \text{max\_sector\_ratio} \rfloor)$ | 超額配對排隊不進場 |
| 5 | **方向冷卻 Cooldown** | 停損後或強平後 | 等 Z-Score 穿越 0 才解凍 |
| 6 | **波動率自適應 VOL ADJ** | `use_vol_adjust=True` | 以 20 日滾動 $\sigma$ 放大基準 $\sigma_{form}$ |

### PSL（全域組合停損）正確 PnL 計帳邏輯

PSL 觸發時，開放部位尚未平倉，`Trade_PnL = 0`（僅在平倉時記帳）。因此必須使用 `Unrealized_PnL` 計算最終虧損：

$$\text{final\_realized} = \text{Realized\_PnL}_\text{before\_stop} + \text{Unrealized\_PnL}_\text{at\_stop}$$

若錯誤使用 `Trade_PnL`（平倉前為 0），PSL 觸發時開放部位的損失將永遠不被記帳，造成資金計算失真。

## 🤖 三、 DRL 門檻選擇式交易邏輯 v4 (`drl_threshold_trading.py`)

> **策略 #5、#6、#8、#10 使用此模組。**

### 3.1 演進脈絡：為何是「門檻選擇」而非「逐日定位」

三代逐日定位動作空間的 DRL（v1 online DQN `drl_lstm_trading.py`、v2 修復版 `drl_lstm_v2_trading.py`、v3 FQI `drl_fqi_trading.py`，全數已封存）已被系統性證偽：即使逐步修復訓練的計算與統計缺陷（v1→v2 修復假共享/獎勵重複計算/epsilon 排程；v2→v3 換用 Fitted Q-Iteration 修復訓練效率），「每天自由決定持倉」的動作空間仍讓模型對日級噪音計時，OOS 換手費用高達基準的 6 倍以上（中位 Sharpe −1.1~−2.3 vs Z-Score ≈ 0）。失敗根因被隔離在**動作空間設計本身**，而非訓練方法。

v4 徹底縮減動作空間：agent **每配對每期只做「一個」決策**——

$$\text{動作選單（9）} = \{\text{SKIP（不交易）}\} \cup \{(entry_z, exit_z) : entry_z \in \{1.5,2.0,2.5,3.0\},\ exit_z \in \{0.0,0.5\}\}$$

選定後交給標準 Z-Score 狀態機（`_fast_threshold_pnl`，與 `zscore_trading.py` 邏輯完全同構）執行整個交易期。

### 3.2 結構保證

1. 選單包含靜態基準 $(2.0, 0.0)$ → 策略空間 $\supseteq$ Z-Score 基準
2. 訓練樣本不足（走勢初期）時自動選基準動作 → 早期 $\equiv$ Z-Score
3. SKIP 讓 agent 可以拒絕壞配對（Z-Score 做不到的選擇性）

### 3.3 學習問題的性質：全資訊監督回歸，非探索型 Bandit

歷史配對期的「全部 9 個動作的報酬」都可以精確**反事實回算**（對交易期價格逐一模擬 9 組門檻）——不需要實際執行動作才能觀察報酬，因此是全資訊監督回歸問題，無探索-利用問題，樣本效率最高。

$$\text{net}: \text{MLP}(12\text{ 維形成期特徵}) \rightarrow 9\text{ 個動作的預期報酬}$$

### 3.4 12 維形成期特徵

| # | 特徵 | 計算方式 |
| :---: | :--- | :--- |
| 1 | 期末 Z-Score | $\text{clip}(z_{-1}/3, -3, 3)$ |
| 2 | 期末 \|Z-Score\| | $\text{clip}(\|z_{-1}\|/3, 0, 3)$ |
| 3 | 零穿越頻率 | $\text{clip}(zc \times 10, 0, 3)$ |
| 4 | 均值回歸速度 | $\ln(\text{halflife})/3$ |
| 5 | 近期 Z 波動 regime | 近 21 日 std / 全期 std $- 1$ |
| 6 | 近期 Z 趨勢 | $(z_{-1} - \bar{z}_{21})/3$ |
| 7 | 兩股相關係數 | $\text{corr}(\ln P_A, \ln P_B)$ |
| 8 | 波動率比 | $\text{clip}(\sigma_A/\sigma_B - 1, -3, 3)$ |
| 9 | 對沖比例偏移 | $\text{clip}(\beta - 1, -3, 3)$ |
| 10 | Spread 振幅 | $\text{clip}(\sigma_\epsilon \times 5, 0, 3)$ |
| 11 | 形成期 \|Z\|>2 佔比 | $\text{mean}(\|z\|>2)$ |
| 12 | 形成期最大 \|Z\| | $\text{clip}(\max\|z\|/5, 0, 3)$ |

### 3.5 Walk-Forward 訓練與無前視保證

$$\text{eligible} = \{(f, r, t_e) \in \text{buffer} : t_e < \text{trade\_start}_k\}$$

期 $k$ 決策只用「交易期已於 $k$ 開始前結束」的樣本訓練——與形成期 `ml_pair_quality.py`（已封存）的 walk-forward 規則逐字一致。樣本數 $\ge$ `thr_min_train_samples`（預設 200）才啟用網路決策，否則 fallback 至基準動作 $(2.0, 0.0)$。

`_shared` 狀態以 `variant_id`（含策略名稱、Top_n、停損、MSR）為 key 隔離，避免同一 worker process 生命週期內依序處理的不同策略變體共用同一份網路/緩衝區、互相污染訓練資料（此為本專案曾發生並修復的真實 bug，見 commit `9ae56ec`；修復後以完整 30 組變體重跑驗證，結論未變）。

### 3.6 ThresholdNet 網路架構

```
輸入 (12 維) → Linear(64) → ReLU → Linear(64) → ReLU → Linear(9) → 9 個動作的預期報酬
```

訓練：`thr_train_epochs=40`，`Adam(lr=1e-3)`，`MSELoss`，batch size $\min(4096, n_\text{eligible})$。

In [ ]:
"""
drl_threshold_trading.py — 動作選單與反事實標籤生成邏輯示範
"""
import numpy as np

# 9 個動作：index 0 = SKIP；1-8 = (entry_z, exit_z) 組合
ACTIONS = [None] + [(ez, xz) for ez in (1.5, 2.0, 2.5, 3.0) for xz in (0.0, 0.5)]
BASELINE_IDX = ACTIONS.index((2.0, 0.0))
print(f"動作選單（{len(ACTIONS)} 個）：{ACTIONS}")
print(f"基準動作 index = {BASELINE_IDX}（對應標準 Z-Score 進出場 entry_z=2.0, exit_z=0.0）")

# walk-forward 資格判斷示範：只用「交易期已於本期開始前結束」的歷史樣本
buffer = [
    ("feat_A", "label_A", np.datetime64("2020-03-15")),  # 交易期結束於 2020-03-15
    ("feat_B", "label_B", np.datetime64("2020-06-20")),
    ("feat_C", "label_C", np.datetime64("2020-09-25")),
]
trade_start_k = np.datetime64("2020-07-01")  # 本期交易期起始日

eligible = [(f, l) for f, l, te in buffer if te < trade_start_k]
print(f"\n本期 trade_start={trade_start_k}：{len(eligible)}/{len(buffer)} 筆歷史樣本符合 walk-forward 資格")
for f, l in eligible:
    print(f"  可用於訓練：{f} -> {l}")
print("（第 3 筆 2020-09-25 結束的樣本被正確排除，因為它在本期交易開始『之後』才產生——防止前視）")


## 📊 四、 交易期模組特徵對比總結

| 交易特徵 | Z-Score 狀態機 (`zscore_trading`) | DRL 門檻選擇式 v4 (`drl_threshold_trading`) |
| :--- | :---: | :---: |
| **使用策略** | #1–#4、#7、#9（共 6 個） | #5、#6、#8、#10（共 4 個） |
| **決策粒度** | 每日 Z-Score 訊號 | 每配對每期選 1 個門檻組合（含 SKIP） |
| **開倉條件** | $\|Z_t\| >$ `entry_z` = 2.0 突破建倉 | 選定門檻的 $entry_z$ 突破建倉 |
| **平倉條件** | $\|Z_t\| \le$ `exit_z` = 0.0 回歸均值 | 選定門檻的 $exit_z$ 回歸 |
| **對沖比例** | OLS $\beta$ 風險中性加權 | 同左（沿用相同公式） |
| **學習方式** | 無（規則型） | Walk-forward 監督回歸（12 維特徵 → 9 動作預期報酬） |
| **策略空間關係** | — | $\supseteq$ Z-Score（選單含基準動作，訓練不足時 fallback） |
| **風控支援** | SL / DSZ / PSL / MSR / Cooldown / VOL ADJ | 相同六項風控機制對應套用於選定門檻後的模擬 |

**Spread 空間**：兩者對同一組配對使用完全相同的 `Log_Mean/Std_A/B` 標準化空間與 Z-Score 公式（見上方一、1.1 節），確保「Z-Score 基準 vs DRL 疊加」的比較中，唯一變因是交易決策邏輯，而非 spread 定義的差異。

## 🏆 五、 現役策略績效總比較（`results/result.db` `strategy_summaries`）

以下數據截至 2026-07-04，涵蓋全部 10 個現役策略在 `top_n_list × stop_loss_list`（部分策略另含 `entry_z_list`/`dynamic_stop_z_list`）網格搜尋下的完整回測結果。**評估目標**（源自研究提問）：是否存在「機器學習配對 + 深度學習交易」策略，績效優於 SSD/DTW 距離法基準，且年化報酬 $\ge 2\%$。

### 5.1 各策略最佳 Sharpe 組合

| 排名 | 策略 | 交易端 | 最佳組合 | Sharpe | 年化報酬 | 最大回撤 | 勝率 |
| :---: | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| 1 | SSD-DTW-PCA (Paper-Fixed) | Z-Score | Top3, SL0% | **0.558** | 2.11% | −9.38% | 32.6% |
| 2 | HDBSCAN (Cluster-SSD-DTW-PCA-PCA5-DRL-THR) | DRL | Top10, SL0% | 0.536 | 1.30% | −7.82% | 37.5% |
| 3 | HDBSCAN (Cluster-SSD-DTW-PCA-DRL-THR) | DRL | Top10, SL15% | 0.417 | 0.85% | −8.77% | 37.1% |
| 4 | Agglomerative (Fundamentals-DRL-THR) | DRL | Top1, SL5% | 0.362 | **3.03%** | −23.83% | 46.6% |
| 5 | Agglomerative (Fundamentals) | Z-Score | Top1, SL15% | 0.347 | 2.17% | −17.17% | 35.1% |
| 6 | SSD (Rolling-DRL-THR) | DRL | Top10, SL0% | 0.346 | 0.70% | −5.31% | 35.8% |
| 7 | HDBSCAN (Cluster-SSD-DTW-PCA-PCA5) | Z-Score | Top10, SL0% | 0.294 | 0.90% | −11.69% | 33.9% |
| 8 | DTW (Paper-Fixed) | Z-Score | Top3, SL0% | 0.271 | 1.06% | −11.92% | 32.8% |
| 9 | SSD (Rolling) | Z-Score | Top1, SL0% | 0.262 | 2.30% | −23.61% | 34.4% |
| 10 | HDBSCAN (Cluster-SSD-DTW-PCA) | Z-Score | Top5, SL0% | 0.218 | 0.69% | −19.75% | 33.9% |

### 5.2 各策略最佳年化報酬組合

| 排名 | 策略 | 交易端 | 最佳組合 | Sharpe | 年化報酬 |
| :---: | :--- | :---: | :---: | :---: | :---: |
| 1 | **Agglomerative (Fundamentals-DRL-THR)** | DRL | Top1, SL5% | 0.362 | **3.03%** |
| 2 | Agglomerative (Fundamentals) | Z-Score | Top1, SL0% | 0.325 | 2.70% |
| 3 | SSD (Rolling-DRL-THR) | DRL | Top1, SL0% | 0.297 | 2.64% |
| 4 | SSD (Rolling) | Z-Score | Top1, SL0% | 0.262 | 2.30% |
| 5 | SSD-DTW-PCA (Paper-Fixed) | Z-Score | Top3, SL0% | 0.558 | 2.11% |
| 6 | HDBSCAN (Cluster-SSD-DTW-PCA-DRL-THR) | DRL | Top1, SL0% | 0.264 | 1.63% |
| 7 | HDBSCAN (Cluster-SSD-DTW-PCA-PCA5-DRL-THR) | DRL | Top10, SL0% | 0.536 | 1.30% |
| 8 | HDBSCAN (Cluster-SSD-DTW-PCA-PCA5) | Z-Score | Top1, SL15% | 0.213 | 1.08% |
| 9 | DTW (Paper-Fixed) | Z-Score | Top3, SL0% | 0.271 | 1.06% |
| 10 | HDBSCAN (Cluster-SSD-DTW-PCA) | Z-Score | Top1, SL0% | 0.148 | 0.82% |

### 5.3 結論：是否達成「ML 配對 + DRL 交易 ≥ 2% 年化 且優於 SSD/DTW」？

**部分達成，且已知有前視限制**。5 個策略跨過 2% 年化門檻，由高至低：Agglomerative Fundamentals DRL THR (3.03%) > Agglomerative Fundamentals (2.70%) > SSD Rolling DRL THR (2.64%) > SSD Rolling (2.30%) > SSD-DTW-PCA Paper Fixed (2.11%)。

- **Agglomerative Fundamentals（分組消融第三支）是全組唯一的「ML 配對」方法**，在 Sharpe（0.35 vs 0.26）與年化報酬（2.70% vs 2.30%）雙指標上同時優於 SSD (Rolling) 距離法基準；疊加 DRL-THR 後年化報酬進一步提升至全組最高的 3.03%（Sharpe 0.36，DRL 對 Sharpe 的提升幅度不如 #4→#6、#7→#8 的疊加效果明顯）。
  ⚠️ **但此結果受限於基本面前視偏誤**：市值/本益比為單一時點靜態快照套用於全部 2000–2025 歷史窗口（見 `notebooks/formation.ipynb` 4.5 節），無法在不取得歷史逐日基本面資料（WRDS/Compustat 等付費源）前完全排除此限制對早期窗口績效的貢獻。
- **HDBSCAN Cluster SSD-DTW-PCA 系列（PCA5 + DRL-THR）驗證了「DRL 交易端能穩定提升配對訊號的風險調整後報酬」**：Sharpe 從 Z-Score 基準的 0.29 提升到 DRL-THR 的 0.54（全組第二高），但年化報酬僅 1.30%，未跨過門檻——瓶頸在**配對訊號本身**，DRL 交易端補不齊原始年化報酬的差距。
- **SSD-DTW-PCA (Paper-Fixed)（非 ML 方法，GICS 靜態分組 + 距離排序）仍是全組 Sharpe 最高者（0.56）**，且是唯一同時滿足「Sharpe 最佳」與「年化 ≥2%」的距離法基準——ML 分組消融（HDBSCAN／Agglomerative）目前尚未在乾淨對照（相同排序邏輯、相同交易端）下全面超越它。
- 三次 HDBSCAN 系列迭代（PCA-Loadings 原始版 → Cluster-SSD-DTW-PCA → PCA5 降維修復版，皆已於 `notebooks/formation.ipynb` 詳述）與兩次 ML Pair Quality 監督式排序嘗試（已封存）均未能讓 ML 分組消融在誠實對照下超越 SSD/DTW 距離法基準的年化報酬天花板；本次 Agglomerative Fundamentals 是唯一的例外，但需連同其前視限制一併解讀。

## 📖 參考文獻

本研究交易期模組直接引自以下論著：

---

### Gatev, Goetzmann & Rouwenhorst (2006) — Z-Score 交易框架基礎

> **論著：** Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative value arbitrage rule. *Review of Financial Studies*, **19**(3), 797–827.

**論著核心貢獻（交易期部分）：**

- Z-Score 突破 $\pm 2\sigma$ 進場、回歸均值（$Z=0$）出場是配對交易的基準實作
- 靜態形成期統計在整個交易期保持不變（本研究 `zscore_window=0`）
- 等市值/風險中性對沖消除市場 Beta 暴露；交易成本對長期獲利能力有顯著侵蝕

**本研究對應：** `zscore_trading.py` 靜態 Z-Score 狀態機（現役策略 #1–#4、#7、#9 直接使用；#5、#6、#8、#10 的基準動作與風控外層亦源自此框架）

---

### Krauss, Do & Huck (2016) — 交易摩擦成本與風控設計依據

> **論著：** Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies: Distance, cointegration and copula methods. *European Journal of Operational Research*.

**論著核心貢獻：**

- 交易成本（手續費 + 滑點）對配對交易獲利能力有顯著侵蝕，必須納入回測假設
- 停損機制可降低最大回撤，但需權衡觸發頻率；產業分散限制避免系統性風險集中

**本研究對應：** `fee_rate=slippage_rate=0.001`（各 0.1%），六大風控機制設計（SL/DSZ/PSL/MSR/Cooldown/VOL ADJ）

---

### Kim & Kim (2019) — 門檻選擇式強化學習配對交易

> **論著：** Kim, T., & Kim, H. Y. (2019). Optimizing the pairs-trading strategy using deep reinforcement learning with trading and stop-loss boundaries. *Complexity*, 2019, Article 3582516.

**論著核心貢獻：**

- 提出讓 RL agent **輸出交易門檻（進場/停損邊界）而非逐日持倉動作**的配對交易框架
- Spread 觸及交易門檻並回歸均值時給予正獎勵，觸及停損門檻或未能回歸時給予負獎勵
- 相較逐日決策，門檻選擇式大幅降低動作空間維度，緩解過度交易與雜訊過擬合問題

**本研究對應：** `drl_threshold_trading.py`（v4）的核心設計理念——9 個 (entry_z, exit_z) 門檻組合 + SKIP 的動作選單，是本研究 v1–v3 逐日定位動作空間系統性失敗後，改採此文獻風格重新設計的成果

---

### Mnih et al. (2015) — DQN 深度強化學習框架

> **論著：** Mnih, V., Kavukcuoglu, K., Silver, D., et al. (2015). Human-level control through deep reinforcement learning. *Nature*, **518**, 529–533.

**論著核心貢獻：**

- 提出深度 Q 網路（DQN），以深層神經網路逼近動作值函數；Experience Replay 打破時序相關性；Target Network 穩定訓練

**本研究對應：** 已封存 v1（`drl_lstm_trading.py`）/ v2（`drl_lstm_v2_trading.py`）的 DQN 訓練框架起點；v4（現役）改用反事實監督回歸取代線上 DQN，但問題的 MDP 建模仍源自此框架

---

### Sutton & Barto (2018) — 強化學習理論框架

> **論著：** Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.

**論著核心貢獻：**

- 馬可夫決策過程（MDP）$(S, A, P, R, \gamma)$ 框架；探索-利用權衡的理論基礎

**本研究對應：** v4 DRL-THR 的問題建模——但因動作空間內全部選項的反事實報酬皆可精確回算，v4 實際上退化為**全資訊監督回歸**而非需要探索的強化學習問題，這是 v4 相較 v1–v3 樣本效率大幅提升的理論原因